In [1]:
import os
import json
import uuid
import oracledb
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_oracledb.vectorstores.oraclevs import OracleVS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores.utils import DistanceStrategy

d:\IE103_Final_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
username = "system"
password = "oracle"
dsn = "192.168.1.248:1521/FREEPDB1"

In [3]:
try:
    connection = oracledb.connect(user=username, password=password, dsn=dsn)
    print("Connection successful!")

except oracledb.Error as e:
    error_obj, = e.args
    print(f"Oracle Error: {error_obj.message}")

except Exception as e:
    print(f"Undefined Error: {e}")

Connection successful!


In [4]:
corpus_path = os.path.join(os.path.dirname(os.getcwd()), "scifact", "corpus.jsonl")
corpus_path

'd:\\IE103_Final_Project\\scifact\\corpus.jsonl'

In [5]:
documents_langchain = []

with open(corpus_path, "r", encoding="utf-8") as document_jsonl_list:
    for line in document_jsonl_list:
        doc = json.loads(line)
        metadata = {"id": doc["_id"], "title": doc["title"]}
        doc_langchain = Document(page_content=doc["text"], metadata=metadata)
        documents_langchain.append(doc_langchain)

In [6]:
len(documents_langchain)

5183

In [7]:
documents_langchain[0]

Document(metadata={'id': '4983', 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.'}, page_content='Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess effects of prematurity on cerebral white matter development, early gestation preterm infants (n = 10) were studied a second time at term. In the central white matter the mean apparent diffusion coefficient at 28 wk was high, 1.8 microm2/ms, and decreased toward term to 1.2 microm2/ms. In the posterior limb of th

In [8]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
)

In [9]:
chunks_langchain = text_splitter.split_documents(documents_langchain)

In [10]:
len(chunks_langchain)

18168

In [11]:
chunks_langchain[0]

Document(metadata={'id': '4983', 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.'}, page_content='Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess')

In [12]:
for chunk in chunks_langchain:
    chunk.metadata["corpus_id"] = chunk.metadata.get("id")
    chunk.metadata["id"] = str(uuid.uuid4())

In [13]:
chunks_langchain[0]

Document(metadata={'id': 'b38636a1-de8a-4b56-957c-af15de7953f8', 'title': 'Microstructural development of human newborn cerebral white matter assessed in vivo by diffusion tensor magnetic resonance imaging.', 'corpus_id': '4983'}, page_content='Alterations of the architecture of cerebral white matter in the developing human brain can affect cortical development and result in functional disabilities. A line scan diffusion-weighted magnetic resonance imaging (MRI) sequence with diffusion tensor analysis was applied to measure the apparent diffusion coefficient, to calculate relative anisotropy, and to delineate three-dimensional fiber architecture in cerebral white matter in preterm (n = 17) and full-term infants (n = 7). To assess')

In [14]:
model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4000.53it/s]


In [15]:
BATCH_SIZE = 500

vector_store = None

for i in range(0, len(chunks_langchain), BATCH_SIZE):
    batch = chunks_langchain[i:min(i+BATCH_SIZE, len(chunks_langchain))]

    if vector_store is None:
        vector_store = OracleVS.from_documents(
            batch,
            model,
            client=connection,
            table_name="SciFact",
            distance_strategy=DistanceStrategy.COSINE,
        )
    else:
        vector_store.add_documents(batch)

    print(f"Done {i + len(batch)}/{len(chunks_langchain)}")

Done 500/18168
Done 1000/18168
Done 1500/18168
Done 2000/18168
Done 2500/18168
Done 3000/18168
Done 3500/18168
Done 4000/18168
Done 4500/18168
Done 5000/18168
Done 5500/18168
Done 6000/18168
Done 6500/18168
Done 7000/18168
Done 7500/18168
Done 8000/18168
Done 8500/18168
Done 9000/18168
Done 9500/18168
Done 10000/18168
Done 10500/18168
Done 11000/18168
Done 11500/18168
Done 12000/18168
Done 12500/18168
Done 13000/18168
Done 13500/18168
Done 14000/18168
Done 14500/18168
Done 15000/18168
Done 15500/18168
Done 16000/18168
Done 16500/18168
Done 17000/18168
Done 17500/18168
Done 18000/18168
Done 18168/18168
